In [0]:
# ════════════════════════════════════════════
# Workshop Configuration — no edits needed
# ════════════════════════════════════════════
import re

# Read the current user directly from the Databricks workspace context
_raw_user   = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
USERNAME    = re.sub(r'[^a-zA-Z0-9]', '_', _raw_user.split("@")[0])

CATALOG     = "workshop"         # shared catalog, pre-configured by facilitator
SCHEMA      = USERNAME           # your personal schema inside the catalog

# MLflow experiment path — lives in your personal workspace folder
EXPERIMENT_PATH = f"/Users/{_raw_user}/gdm_yield_{USERNAME}"

print(f"📍 Logged in as  : {_raw_user}")
print(f"📍 USERNAME       : {USERNAME}")
print(f"📍 Your namespace : {CATALOG}.{SCHEMA}")
print(f"📍 Your table     : {CATALOG}.{SCHEMA}.yield_trials")
print(f"📍 Your model     : gdm-yield-model-{USERNAME}")
print(f"📍 MLflow path    : {EXPERIMENT_PATH}")

In [0]:
# Create your personal schema (run once)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print(f"✅ Schema ready: {CATALOG}.{SCHEMA}")

In [0]:
# ════════════════════════════════════════════
# Install Kaggle CLI and download dataset
# ════════════════════════════════════════════
# Note: Kaggle credentials must be configured first
# Option 1: Set environment variables KAGGLE_USERNAME and KAGGLE_KEY
# Option 2: Upload kaggle.json to ~/.kaggle/kaggle.json

%pip install kaggle --quiet

import os
import zipfile

# Download the crop yield dataset
print("📥 Downloading dataset from Kaggle...")
!kaggle datasets download -d patelris/crop-yield-prediction-dataset

# Extract the ZIP file
print("📦 Extracting files...")
with zipfile.ZipFile('crop-yield-prediction-dataset.zip', 'r') as zip_ref:
    zip_ref.extractall('.')

print("✅ Dataset downloaded and extracted successfully")
print(f"📂 Files: {os.listdir('.')}")

In [0]:
# ════════════════════════════════════════════
# Load CSV into Spark DataFrame and explore
# ════════════════════════════════════════════
import pandas as pd

# Load CSV with pandas first (works on serverless)
print("📂 Loading yield_df.csv with pandas...\n")
pandas_df = pd.read_csv("yield_df.csv")

# Convert to Spark DataFrame
df = spark.createDataFrame(pandas_df)

print("═" * 60)
print("📊 DATASET SCHEMA")
print("═" * 60)
df.printSchema()

print("\n" + "═" * 60)
print(f"📈 TOTAL ROWS: {df.count():,}")
print("═" * 60)

print("\n" + "═" * 60)
print("🔍 SAMPLE DATA (5 rows)")
print("═" * 60)
display(df.limit(5))

print("\n" + "═" * 60)
print("🌾 UNIQUE CROP TYPES (Item column)")
print("═" * 60)
items = df.select("Item").distinct().orderBy("Item").collect()
for row in items:
    print(f"  • {row['Item']}")

print("\n" + "═" * 60)
print("🌎 AREAS INCLUDING 'Argentina' or 'Brazil'")
print("═" * 60)
areas = df.filter(
    (df.Area.contains("Argentina")) | (df.Area.contains("Brazil"))
).select("Area").distinct().orderBy("Area").collect()
for row in areas:
    print(f"  • {row['Area']}")

print("\n✅ Dataset exploration complete")

In [0]:
# ════════════════════════════════════════════
# Save DataFrame as Delta table
# ════════════════════════════════════════════

table_name = f"{CATALOG}.{SCHEMA}.yield_trials"

print(f"💾 Saving DataFrame to: {table_name}\n")

# Drop the index column and clean column names
df_clean = df.drop("Unnamed: 0")

# Replace invalid characters in column names
for col_name in df_clean.columns:
    clean_name = col_name.replace("/", "_").replace(" ", "_")
    if clean_name != col_name:
        df_clean = df_clean.withColumnRenamed(col_name, clean_name)
        print(f"➡️  Renamed: '{col_name}' → '{clean_name}'")

df_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(table_name)

print(f"\n✅ Table saved successfully: {table_name}")
print(f"   Columns: {', '.join(df_clean.columns)}")

In [0]:
# ════════════════════════════════════════════
# Verification Queries
# ════════════════════════════════════════════

table_name = f"{CATALOG}.{SCHEMA}.yield_trials"

print("═" * 60)
print("📊 COUNT VERIFICATION")
print("═" * 60)

# Count total rows
count_df = spark.sql(f"""
    SELECT COUNT(*) as total_rows 
    FROM {table_name}
""")

display(count_df)

print("\n" + "═" * 60)
print("🔍 SAMPLE DATA VERIFICATION")
print("═" * 60)

# Sample first 5 rows
sample_df = spark.sql(f"""
    SELECT * 
    FROM {table_name} 
    LIMIT 5
""")

display(sample_df)

print("\n✅ Verification complete")

In [0]:
# ════════════════════════════════════════════
# Install ML Dependencies
# ════════════════════════════════════════════
%pip install xgboost --quiet

In [0]:
# ════════════════════════════════════════════
# Build ML Pipeline with Integrated Preprocessing
# ════════════════════════════════════════════
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

# Load table into pandas DataFrame
table_name = f"{CATALOG}.{SCHEMA}.yield_trials"
print(f"📥 Loading table: {table_name}\n")

df_pandas = spark.sql(f"SELECT * FROM {table_name}").toPandas()

print(f"✅ Loaded {len(df_pandas):,} rows\n")

# Define target and features
target_col = "hg_ha_yield"  # Note: renamed from "hg/ha_yield"
feature_cols = [
    "Item", 
    "Area", 
    "average_rain_fall_mm_per_year", 
    "pesticides_tonnes", 
    "avg_temp", 
    "Year"
]

categorical_features = ["Item", "Area"]
numerical_features = [
    "average_rain_fall_mm_per_year", 
    "pesticides_tonnes", 
    "avg_temp", 
    "Year"
]

X = df_pandas[feature_cols]
y = df_pandas[target_col]

print("═" * 60)
print("🏗️  BUILDING SKLEARN PIPELINE")
print("═" * 60)

# Build preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=-1
            ),
            categorical_features
        ),
        (
            "numerical",
            "passthrough",
            numerical_features
        )
    ]
)

# Complete pipeline with model
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBRegressor(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.1,
        random_state=42
    ))
])

print("  ✓ ColumnTransformer configured:")
print(f"    • OrdinalEncoder on: {categorical_features}")
print(f"    • Passthrough on: {numerical_features}")
print("  ✓ XGBRegressor (n_estimators=100, max_depth=5, lr=0.1)")

print("\n" + "═" * 60)
print("✂️  TRAIN/TEST SPLIT (80/20)")
print("═" * 60)

# Split data (keeping raw strings in X_train/X_test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(f"\n📊 Dataset Dimensions:")
print(f"  • X_train: {X_train.shape}")
print(f"  • X_test:  {X_test.shape}")
print(f"  • y_train: {y_train.shape}")
print(f"  • y_test:  {y_test.shape}")

print(f"\n✅ Pipeline ready for training")

In [0]:
# ════════════════════════════════════════════
# Train Pipeline and Register with MLflow
# ════════════════════════════════════════════
import mlflow
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Configure MLflow experiment
print("═" * 60)
print("🔬 CONFIGURING MLFLOW EXPERIMENT")
print("═" * 60)
mlflow.set_experiment(EXPERIMENT_PATH)
print(f"✓ Experiment set to: {EXPERIMENT_PATH}\n")

# Train the pipeline
print("═" * 60)
print("🚀 TRAINING PIPELINE")
print("═" * 60)
print("Training XGBoost with integrated preprocessing...\n")

pipeline.fit(X_train, y_train)
print("✅ Pipeline trained successfully\n")

# Make predictions
y_pred = pipeline.predict(X_test)

# Calculate metrics
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("═" * 60)
print("📊 MODEL EVALUATION METRICS")
print("═" * 60)
print(f"  • RMSE: {rmse:,.2f}")
print(f"  • R²:   {r2:.4f}\n")

# Register model with MLflow
print("═" * 60)
print("📦 REGISTERING MODEL WITH MLFLOW")
print("═" * 60)

model_name = f"{CATALOG}.{SCHEMA}.gdm-yield-model-{USERNAME}"

# Infer model signature (required for Unity Catalog)
signature = mlflow.models.infer_signature(X_train, pipeline.predict(X_train))

with mlflow.start_run() as run:
    # Log parameters
    mlflow.log_params({
        "n_estimators": 100,
        "max_depth": 5,
        "learning_rate": 0.1
    })
    
    # Log metrics
    mlflow.log_metrics({
        "rmse": rmse,
        "r2": r2
    })
    
    # Log the complete pipeline (preprocessing + model) with signature
    mlflow.sklearn.log_model(
        sk_model=pipeline,
        artifact_path="gdm_yield_pipeline",
        registered_model_name=model_name,
        signature=signature
    )
    
    run_id = run.info.run_id
    print(f"✓ Run ID: {run_id}")
    print(f"✓ Model registered: {model_name}")
    print(f"✓ Artifact path: gdm_yield_pipeline")
    print(f"✓ Signature included: {signature}\n")

print("═" * 60)
print("📈 GENERATING VISUALIZATIONS")
print("═" * 60)

# 1. Feature Importances Bar Chart
feature_names = [
    "Tipo de Cultivo",
    "Pais",
    "Lluvia (mm/ano)",
    "Pesticidas (t)",
    "Temp Prom (°C)",
    "Ano"
]

importances = pipeline.named_steps["model"].feature_importances_

# Sort by importance
sorted_idx = np.argsort(importances)[::-1]
sorted_importances = importances[sorted_idx]
sorted_names = [feature_names[i] for i in sorted_idx]

fig1 = go.Figure([
    go.Bar(
        x=sorted_importances,
        y=sorted_names,
        orientation='h',
        marker=dict(color='steelblue')
    )
])

fig1.update_layout(
    title="Importancia de Features (XGBoost)",
    xaxis_title="Importancia",
    yaxis_title="Feature",
    height=400,
    showlegend=False
)

print("\n📊 Feature Importances:")
fig1.show()

# 2. Predicted vs Actual Scatter Plot
fig2 = go.Figure()

# Scatter plot
fig2.add_trace(go.Scatter(
    x=y_test,
    y=y_pred,
    mode='markers',
    marker=dict(color='steelblue', size=5, opacity=0.6),
    name='Predicciones'
))

# Perfect prediction line (diagonal)
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())

fig2.add_trace(go.Scatter(
    x=[min_val, max_val],
    y=[min_val, max_val],
    mode='lines',
    line=dict(color='red', dash='dash', width=2),
    name='Predicción Perfecta'
))

fig2.update_layout(
    title=f"Rendimiento Predicho vs Real (R² = {r2:.4f})",
    xaxis_title="Rendimiento Real (hg/ha)",
    yaxis_title="Rendimiento Predicho (hg/ha)",
    height=500,
    showlegend=True
)

print("\n📊 Predicted vs Actual:")
fig2.show()

print("\n✅ Model training and registration complete!")